# Workshop Project 2: LLM Text Watermarking — KGW Method

Can we secretly tag AI-generated text so it's **detectable** but **invisible** to readers?

This notebook implements the **KGW (Kirchenbauer et al.)** watermarking scheme:
1. **Embed** a statistical watermark during text generation
2. **Detect** it using a Z-score test (no model weights needed!)
3. **Attack** it with semantic paraphrase rewriting
4. **Visualize** token-level greenlist highlighting and Z-score comparisons



### How KGW Works (intuition)

For each token position, the previous token determines a **pseudorandom split** of the vocabulary into:
- 🟩 **Greenlist** (γ fraction) — these tokens get a logit boost (+δ)
- 🟥 **Redlist** (1−γ fraction) — no boost

A watermarked text has **abnormally many greenlist tokens**. The Z-score measures this deviation.

## 0. Setup

In [ ]:
# Install required libraries and check PyTorch + GPU availability
!pip -q install -U transformers accelerate sentencepiece

import torch
torch.backends.cuda.enable_cudnn_sdp(False)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 1. Load Language Model

We use **OPT-350M**, a small causal language model that runs comfortably on Colab (even without GPU).OPT-350M is a small, open-source language model released by Meta. “OPT” stands for Open Pretrained Transformer, and “350M” means the model has 350 million parameters.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "facebook/opt-350m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Force loading with safetensors to bypass torch v2.6 requirement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_safetensors=True
).to(device)

print(f"✅ Model loaded: {model_name}")
print(f"   Vocab size: {tokenizer.vocab_size:,}")

## 2. Configure KGW Watermark

Two key parameters control the watermark:

| Parameter | Symbol | Effect |
|-----------|--------|--------|
| `greenlist_ratio` | γ | Fraction of vocabulary in the greenlist (smaller = stronger signal, but less fluent) |
| `bias` | δ | Logit boost for greenlist tokens (larger = easier to detect, but may hurt quality) |

We use **γ = 0.25, δ = 2.0** — a common setting that balances detectability and text quality.

In [ ]:
from transformers import WatermarkingConfig

watermarking_config = WatermarkingConfig(
    greenlist_ratio=0.25,   # γ: 25% of vocab is "green"
    bias=2.0,               # δ: logit boost for green tokens
    seeding_scheme="selfhash",
    context_width=1
)

print("✅ Watermark config ready")
print(f"   γ = {watermarking_config.greenlist_ratio}, δ = {watermarking_config.bias}")

## 3. Generate Text — Baseline vs Watermarked

We generate two versions from the **same prompt**:
- **Baseline** — normal sampling (no watermark)
- **Watermarked** — KGW biased sampling

Both use `do_sample=True` because watermarking works by modifying the sampling distribution.

In [ ]:
prompt = "The rapid development of artificial intelligence acts as"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# --- Baseline (no watermark) ---
baseline_ids = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True
)
baseline_text = tokenizer.decode(baseline_ids[0], skip_special_tokens=True)

# --- Watermarked ---
watermarked_ids = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=True,
    watermarking_config=watermarking_config
)
watermarked_text = tokenizer.decode(watermarked_ids[0], skip_special_tokens=True)

print("=" * 60)
print("BASELINE TEXT (no watermark)")
print("=" * 60)
print(baseline_text)
print()
print("=" * 60)
print("WATERMARKED TEXT")
print("=" * 60)
print(watermarked_text)

## 4. Detect Watermark — Z-score Test

The detector **does not need the model weights**. It only needs:
- The watermark config (γ, δ, seeding scheme)
- The tokenized text

It recomputes the greenlist for each token position and counts how many tokens fall in the greenlist. The **Z-score** measures whether this count is statistically unlikely under normal (unwatermarked) text.

**Rule of thumb:** Z > 4 → strong evidence of watermark.

In [ ]:
from transformers import WatermarkDetector

detector = WatermarkDetector(
    model_config=model.config,
    device=device,
    watermarking_config=watermarking_config,
    ignore_repeated_ngrams=True
)

# Detect on baseline
det_base = detector(baseline_ids, return_dict=True)
z_base = float(det_base.z_score[0])

# Detect on watermarked
det_wm = detector(watermarked_ids, return_dict=True)
z_wm = float(det_wm.z_score[0])

print(f"Baseline  Z-score: {z_base:.3f}  →  {'⚠️ Watermark detected' if z_base > 4 else '✅ No watermark'}")
print(f"Watermarked Z-score: {z_wm:.3f}  →  {'✅ Watermark detected!' if z_wm > 4 else '⚠️ Not detected'}")

## 5. Token-Level Greenlist Visualization

This is the most intuitive way to **see** the watermark. We color each token:
- 🟩 **Green** = greenlist token (watermark signal)
- 🟥 **Red** = redlist token

A watermarked text should show **much more green** than a baseline text.

In [ ]:
# This code reconstructs the watermark rule (greenlist vs redlist), labels each generated token, and visualizes whether the watermarked text contains significantly more “green” tokens than normal text.
import hashlib

def get_greenlist_for_token(prev_token_id, vocab_size, gamma, seed_scheme="selfhash"):
    """Reproduce KGW greenlist split for a single token position."""
    # Hash the previous token to get a seed
    seed = int(hashlib.sha256(str(prev_token_id).encode()).hexdigest(), 16) % (2**32)
    rng = np.random.RandomState(seed)
    vocab_perm = rng.permutation(vocab_size)
    greenlist_size = int(gamma * vocab_size)
    return set(vocab_perm[:greenlist_size].tolist())

def colorize_tokens(token_ids, tokenizer, vocab_size, gamma):
    """Return list of (token_str, is_green) tuples."""
    tokens = []
    ids = token_ids.squeeze().tolist()
    for i in range(1, len(ids)):
        prev_id = ids[i - 1]
        curr_id = ids[i]
        greenlist = get_greenlist_for_token(prev_id, vocab_size, gamma)
        is_green = curr_id in greenlist
        token_str = tokenizer.decode([curr_id])
        tokens.append((token_str, is_green))
    return tokens

def display_colored_text(tokens, title, ax):
    """Render tokens as colored boxes on a matplotlib axis."""
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)

    x, y = 0.01, 0.92
    line_height = 0.065
    max_width = 0.98

    for token_str, is_green in tokens:
        color = "#c8e6c9" if is_green else "#ffcdd2"  # light green / light red
        # Estimate width
        char_width = max(len(token_str) * 0.008, 0.015)

        if x + char_width > max_width:
            x = 0.01
            y -= line_height
            if y < 0.02:
                break

        ax.text(x, y, token_str, fontsize=8, fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.15", facecolor=color, edgecolor="none", alpha=0.85),
                verticalalignment="top")
        x += char_width + 0.005

vocab_size = tokenizer.vocab_size

# Colorize both texts
baseline_tokens = colorize_tokens(baseline_ids, tokenizer, vocab_size, watermarking_config.greenlist_ratio)
watermarked_tokens = colorize_tokens(watermarked_ids, tokenizer, vocab_size, watermarking_config.greenlist_ratio)

# Count green fractions
base_green_frac = sum(1 for _, g in baseline_tokens if g) / len(baseline_tokens)
wm_green_frac = sum(1 for _, g in watermarked_tokens if g) / len(watermarked_tokens)

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

display_colored_text(baseline_tokens, f"Baseline — Green fraction: {base_green_frac:.1%}", axes[0])
display_colored_text(watermarked_tokens, f"Watermarked — Green fraction: {wm_green_frac:.1%}", axes[1])

# Legend
green_patch = mpatches.Patch(color="#c8e6c9", label="Greenlist token")
red_patch = mpatches.Patch(color="#ffcdd2", label="Redlist token")
fig.legend(handles=[green_patch, red_patch], loc="lower center", ncol=2, fontsize=11, frameon=False)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.show()

print(f"\nBaseline  green fraction: {base_green_frac:.1%} (expected ~{watermarking_config.greenlist_ratio:.0%} by chance)")
print(f"Watermarked green fraction: {wm_green_frac:.1%} (should be >> {watermarking_config.greenlist_ratio:.0%})")

## 6. Paraphrase Attack

A **paraphrase attack** rewrites the watermarked text to preserve meaning but change the wording — disrupting the greenlist token pattern.

We use a T5-based paraphrase model and generate **10 paraphrase candidates**, then pick the one with the lowest Z-score (strongest attack).

In [ ]:
from transformers import AutoModelForSeq2SeqLM

para_model_name = "Vamsi/T5_Paraphrase_Paws"
para_tokenizer = AutoTokenizer.from_pretrained(para_model_name)
para_model = AutoModelForSeq2SeqLM.from_pretrained(para_model_name).to(device)

print(f"✅ Paraphrase model loaded: {para_model_name}")

In [ ]:
def paraphrase_attack(text: str) -> str:
    """Rewrite text using T5 paraphrase model."""
    enc = para_tokenizer(
        "paraphrase: " + text + " </s>",
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(device)

    outs = para_model.generate(
        **enc,
        max_length=256,
        do_sample=True,
        top_k=120,
        top_p=0.95,
        num_return_sequences=1
    )
    return para_tokenizer.decode(outs[0], skip_special_tokens=True)

def detect_z(text: str) -> float:
    """Compute Z-score for arbitrary text."""
    ids = tokenizer(text, return_tensors="pt").to(device)
    det = detector(ids["input_ids"], return_dict=True)
    return float(det.z_score[0])

# --- Run attack: 5 double paraphrases (stronger rewriting) ---
N = 5
results = []

print("Running double paraphrase attack...")
for i in range(N):
    candidate = paraphrase_attack(paraphrase_attack(watermarked_text))
    z = detect_z(candidate)
    results.append({"trial": i + 1, "z_score": z, "text": candidate})
    print(f"  Trial {i+1}/{N}  z-score = {z:.3f}")

df = pd.DataFrame(results).sort_values("z_score").reset_index(drop=True)

best_text = df.loc[0, "text"]
best_z = float(df.loc[0, "z_score"])

print(f"\n{'='*60}")
print(f"Best attack (lowest z-score): {best_z:.3f}")
print(f"Evaded watermark? (z < 4): {'✅ YES' if best_z < 4 else '❌ NO'}")
print(f"{'='*60}")
print(best_text)

## 7. Results Visualization

### 7a. Z-score Comparison — Three Groups

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Bar chart ---
labels = ["Baseline\n(no watermark)", "Watermarked", "Best\nParaphrase"]
z_values = [z_base, z_wm, best_z]
colors = ["#90a4ae", "#4caf50", "#ff7043"]

bars = axes[0].bar(labels, z_values, color=colors, width=0.5, edgecolor="white", linewidth=1.5)
axes[0].axhline(y=4.0, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Detection threshold (z=4)")

# Add value labels on bars
for bar, val in zip(bars, z_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"z={val:.1f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

axes[0].set_ylabel("Z-score", fontsize=12)
axes[0].set_title("Z-score Comparison", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(axis="y", alpha=0.3)

# --- Right: Scatter plot of all paraphrase trials ---
axes[1].scatter(df["trial"], df["z_score"], s=80, c="#ff7043", edgecolor="white", zorder=3)
axes[1].axhline(y=4.0, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Detection threshold (z=4)")
axes[1].axhline(y=z_wm, color="#4caf50", linestyle="--", linewidth=1.5, alpha=0.7, label=f"Watermarked (z={z_wm:.1f})")

axes[1].set_xlabel("Paraphrase Trial", fontsize=12)
axes[1].set_ylabel("Z-score", fontsize=12)
axes[1].set_title("Paraphrase Attack — Z-score per Trial", fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 7b. Side-by-Side Text Comparison

In [ ]:
print("=" * 70)
print("BASELINE (no watermark)")
print("=" * 70)
print(baseline_text)
print()
print("=" * 70)
print("WATERMARKED")
print("=" * 70)
print(watermarked_text)
print()
print("=" * 70)
print("PARAPHRASED (best attack)")
print("=" * 70)
print(best_text)
print()
print("=" * 70)
print("Z-SCORE SUMMARY")
print("=" * 70)
print(f"  Baseline     : z = {z_base:.3f}  {'(no watermark)' }")
print(f"  Watermarked  : z = {z_wm:.3f}  {'✅ detected' if z_wm > 4 else '⚠️ not detected'}")
print(f"  Paraphrased  : z = {best_z:.3f}  {'✅ still detected' if best_z > 4 else '⚠️ evaded!'}")

## 8. Summary & Takeaways

In this hands-on project you have:

1. **Embedded** a KGW watermark into LLM-generated text by biasing token sampling toward a pseudorandom greenlist

2. **Detected** the watermark using a Z-score test — no model weights required, only the watermark config

3. **Visualized** the watermark at the token level — greenlist highlighting shows where the signal lives

4. **Attacked** the watermark with semantic paraphrasing and measured how the Z-score drops

> **Bottom line:** KGW watermarking is effective and lightweight, but **paraphrase attacks can weaken** the signal. This is an active area of research — how to build watermarks that survive semantic rewriting while keeping text natural.